In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
)
from sklearn.ensemble import RandomForestClassifier


from utils import custom_score, perform_random_search_cv, get_models,score_with_thresh

import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

SEED = 3105
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# numbers of features to test
K_VALUES = [3, 4, 5, 6, 7]
# number of random feature subsets
N_RANDOM = 3
# number of random hyperparameter configs per each model
N_ITER = 3

# fraction of a set we are allowed to contact
CONTACT_RATE = 0.2

## Prepare data

In [2]:
data_dir = Path("../../data")
X = pd.read_csv(data_dir / "x_train.txt", sep=" ")
y = pd.read_csv(data_dir / "y_train.txt", sep=" ").values.ravel()

with open("../feature_selection/selected_features.txt") as f:
    selected = [s.strip().strip("'").strip('"') for s in f.read().split(",")]

X = X[selected]
print(f"X: {X.shape}")

X: (5000, 30)


In [3]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED
)

X_train.reset_index(drop=True, inplace=True)
X_val.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)

print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

train: (3000, 30)  val: (1000, 30)  test: (1000, 30)


## Feature importance and subsets to test

In [4]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
ranking = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(
    ascending=False
)
top15 = ranking.index[:15].tolist()


def random_subsets(k, n, exclude):
    seen = {frozenset(exclude)}
    subsets = []
    while len(subsets) < n:
        s = frozenset(rng.choice(top15, size=k, replace=False))
        if s not in seen:
            seen.add(s)
            subsets.append(sorted(s, key=top15.index))
    return subsets


subsets = {}
for k in K_VALUES:
    topk = ranking.index[:k].tolist()
    topk_1 = ranking.index[1 : k + 1].tolist()
    topk_2 = ranking.index[2 : k + 2].tolist()
    subsets[k] = (
        [("top", topk)]
        + [("top+1", topk_1)]
        + [("top+2", topk_2)]
        + [(f"rand_{i}", s) for i, s in enumerate(random_subsets(k, N_RANDOM, topk))]
    )
    print(f"k={k}: {len(subsets[k])} subsets")

k=3: 6 subsets
k=4: 6 subsets
k=5: 6 subsets
k=6: 6 subsets
k=7: 6 subsets


## Random search CV

In [5]:
results = perform_random_search_cv(
    X_train, y_train, seed=SEED, k_values=K_VALUES, n_iter=N_ITER, subsets=subsets
)

----- Evaluating k = 3 -------
Evaluating subset top
Evaluating subset top+1
Evaluating subset top+2
Evaluating subset rand_0
Evaluating subset rand_1
Evaluating subset rand_2
----- Evaluating k = 4 -------
Evaluating subset top
Evaluating subset top+1
Evaluating subset top+2
Evaluating subset rand_0
Evaluating subset rand_1
Evaluating subset rand_2
----- Evaluating k = 5 -------
Evaluating subset top
Evaluating subset top+1
Evaluating subset top+2
Evaluating subset rand_0
Evaluating subset rand_1
Evaluating subset rand_2
----- Evaluating k = 6 -------
Evaluating subset top
Evaluating subset top+1
Evaluating subset top+2
Evaluating subset rand_0
Evaluating subset rand_1
Evaluating subset rand_2
----- Evaluating k = 7 -------
Evaluating subset top
Evaluating subset top+1
Evaluating subset top+2
Evaluating subset rand_0
Evaluating subset rand_1
Evaluating subset rand_2


/Users/ola/projects/cost-sensitive-marketing/.venv/lib/python3.13/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/ola/projects/cost-sensitive-marketing/.venv/lib/python3.13/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/ola/projects/cost-sensitive-marketing/.venv/lib/python3.13/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warn

In [13]:
res1= pd.read_csv("cv_results_1780264706_seed_3105.csv")
res2= pd.read_csv("cv_results_1780323234_seed_106.csv")

In [16]:
res1['rank_3105'] = res1['cv_score'].rank(ascending=False, method='dense')
res2['rank_106'] = res2['cv_score'].rank(ascending=False, method='dense')
c = res1.join(res2, on=['model', 'best_params'], how="left")

ValueError: len(left_on) must equal the number of levels in the index of "right"

In [17]:
print(res1.columns)
print(res2.columns)

Index(['k', 'kind', 'model', 'cv_score', 'features', 'best_params', 'accuracy',
       'balanced_accuracy', 'precision', 'rank_3105'],
      dtype='str')
Index(['k', 'kind', 'model', 'cv_score', 'features', 'best_params', 'accuracy',
       'balanced_accuracy', 'precision', 'rank_106'],
      dtype='str')


In [15]:
print(res2.index)
print(res2.index.nlevels)

RangeIndex(start=0, stop=240, step=1)
1


In [5]:
results.head(20)

,k,kind,model,cv_score,features,best_params,accuracy,balanced_accuracy,precision
0,3,top,ExtraTrees,585.0,"['V255', 'V191', 'V176']","{'clf__max_depth': 8, 'clf__min_samples_leaf':...",0.610333,0.609856,0.636925
1,3,top+1,ExtraTrees,575.0,"['V191', 'V176', 'V380']","{'clf__max_depth': 11, 'clf__min_samples_leaf'...",0.604000,0.603534,0.628167
2,3,top,XGBoost,565.0,"['V255', 'V191', 'V176']",{'clf__colsample_bytree': np.float64(0.9771170...,0.594333,0.594019,0.608230
3,3,top,GradientBoosting,565.0,"['V255', 'V191', 'V176']",{'clf__learning_rate': np.float64(0.0187075452...,0.607000,0.606652,0.624985
4,3,top+2,ExtraTrees,565.0,"['V176', 'V380', 'V160']","{'clf__max_depth': 5, 'clf__min_samples_leaf':...",0.607667,0.607036,0.645599
5,3,top+2,RandomForest,565.0,"['V176', 'V380', 'V160']","{'clf__max_depth': 9, 'clf__min_samples_leaf':...",0.616667,0.616323,0.636638
6,3,top+1,SVM_RBF,555.0,"['V191', 'V176', 'V380']","{'clf__C': np.float64(4.721229607750618), 'clf...",0.605000,0.604486,0.632550
7,3,top+2,XGBoost,555.0,"['V176', 'V380', 'V160']",{'clf__colsample_bytree': np.float64(0.7648277...,0.604000,0.603586,0.625184
8,3,top,SVM_RBF,550.0,"['V255', 'V191', 'V176']","{'clf__C': np.float64(48.01761562402178), 'clf...",0.610667,0.609923,0.659462
9,3,top+1,XGBoost,550.0,"['V191', 'V176', 'V380']",{'clf__colsample_bytree': np.float64(0.7648277...,0.601333,0.600982,0.620032


In [8]:
results.head(20)

,k,kind,model,cv_score,features,best_params,accuracy,balanced_accuracy,precision
0,3,top,ExtraTrees,580.0,"['V255', 'V191', 'V176']","{'clf__max_depth': 6, 'clf__min_samples_leaf':...",0.611000,0.610521,0.637447
1,3,top+2,XGBoost,570.0,"['V176', 'V380', 'V160']",{'clf__colsample_bytree': np.float64(0.7420988...,0.592667,0.592259,0.610767
2,3,top,GradientBoosting,565.0,"['V255', 'V191', 'V176']",{'clf__learning_rate': np.float64(0.0124474203...,0.614333,0.613986,0.632045
3,3,top,RandomForest,560.0,"['V255', 'V191', 'V176']","{'clf__max_depth': 6, 'clf__min_samples_leaf':...",0.617667,0.617239,0.641139
4,3,top,SVM_RBF,560.0,"['V255', 'V191', 'V176']","{'clf__C': np.float64(0.23522021535538676), 'c...",0.612333,0.611841,0.639095
5,3,top+2,RandomForest,560.0,"['V176', 'V380', 'V160']","{'clf__max_depth': 5, 'clf__min_samples_leaf':...",0.600000,0.599522,0.623639
6,3,top+2,GradientBoosting,555.0,"['V176', 'V380', 'V160']",{'clf__learning_rate': np.float64(0.0124474203...,0.600667,0.600262,0.619405
7,3,rand_2,ExtraTrees,555.0,"[np.str_('V176'), np.str_('V416'), np.str_('V2...","{'clf__max_depth': 9, 'clf__min_samples_leaf':...",0.595000,0.594488,0.619595
8,3,top+1,ExtraTrees,550.0,"['V191', 'V176', 'V380']","{'clf__max_depth': 4, 'clf__min_samples_leaf':...",0.608667,0.607998,0.650054
9,3,top,XGBoost,545.0,"['V255', 'V191', 'V176']",{'clf__colsample_bytree': np.float64(0.7694648...,0.604333,0.604078,0.615296


## Threshold

In [11]:
top15_models = results.iloc[:15]
top15_models
models = get_models()
best_thresholds = []
best_scores = []

for i in range(15):
    k = top15_models['k'].iloc[i]
    features = top15_models['features'].iloc[i]
    params = top15_models['best_params'].iloc[i]
    X_train_tmp = X_train[features]

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', clone(models[top15_models['model'].iloc[i]][0]))
    ])
    pipeline.set_params(**params)
    pipeline.fit(X_train_tmp, y_train)

    proba = pipeline.predict_proba(X_val[features])[:, 1]
    ts = np.linspace(0, 0.7, num=14)
    best_thresh = 0
    best_score = -np.inf
    for t in ts:
        score, contacted = score_with_thresh(y_val, proba, n_var=k, thresh=t)
        if score >= best_score:
            best_score = score
            best_thresh = t
    best_thresholds.append(best_thresh)
    best_scores.append(best_score)

In [12]:
for i in range(15):
    print(best_thresholds[i], best_scores[i])

0.48461538461538456 650
0.5384615384615384 515
0.48461538461538456 530
0.48461538461538456 560
0.5923076923076923 680
0.5923076923076923 605
0.5923076923076923 515
0.5384615384615384 530
0.5384615384615384 545
0.5923076923076923 560
0.5923076923076923 620
0.5384615384615384 575
0.5923076923076923 545
0.5384615384615384 545
0.5923076923076923 545


In [10]:
for i in range(15):
    print(best_thresholds[i], best_scores[i])

0.0 650
0.0 515
0.0 530
0.0 560
0.0 680
0.0 605
0.0 515
0.0 530
0.0 545
0.0 560
0.0 620
0.0 575
0.0 545
0.0 545
0.0 545


In [18]:
pd.read_csv("/Users/ola/projects/cost-sensitive-marketing/src/model/cv_results_1780325436_seed_106.csv")

,k,kind,model,cv_score,features,best_params,accuracy,balanced_accuracy,precision
0,2,rand_1,GradientBoosting,665.0,"[np.str_('V191'), np.str_('V416')]",{'clf__learning_rate': np.float64(0.0186188140...,0.586000,0.585345,0.621537
1,2,rand_1,RandomForest,665.0,"[np.str_('V191'), np.str_('V416')]","{'clf__max_depth': 4, 'clf__min_samples_leaf':...",0.590000,0.589259,0.631340
2,2,top,RandomForest,655.0,"['V255', 'V191']","{'clf__max_depth': 8, 'clf__min_samples_leaf':...",0.564667,0.563976,0.591809
3,2,rand_1,SVM_RBF,655.0,"[np.str_('V191'), np.str_('V416')]","{'clf__C': np.float64(0.23522021535538676), 'c...",0.595000,0.594253,0.638741
4,2,top+1,XGBoost,655.0,"['V191', 'V176']",{'clf__colsample_bytree': np.float64(0.7648277...,0.573667,0.573204,0.592040
5,2,rand_1,LightGBM,655.0,"[np.str_('V191'), np.str_('V416')]",{'clf__learning_rate': np.float64(0.0119796390...,0.578333,0.577784,0.601438
6,2,top+2,RandomForest,645.0,"['V176', 'V380']","{'clf__max_depth': 6, 'clf__min_samples_leaf':...",0.584333,0.583887,0.607415
7,2,top,GradientBoosting,645.0,"['V255', 'V191']",{'clf__learning_rate': np.float64(0.0187075452...,0.570667,0.570180,0.589104
8,2,top,XGBoost,640.0,"['V255', 'V191']",{'clf__colsample_bytree': np.float64(0.7648277...,0.569667,0.569095,0.591096
9,2,rand_1,ExtraTrees,640.0,"[np.str_('V191'), np.str_('V416')]","{'clf__max_depth': 9, 'clf__min_samples_leaf':...",0.588000,0.587290,0.626439
